# Extracting Knowledge from Videos

In this notebook, you'll learn how to **automatically extract knowledge from videos** by converting them to text and cleaning the transcripts using AI. This is a powerful workflow for processing video content at scale.

### The Scenario

Imagine you're a lecturer designing an educational course, and you have a collection of old educational videos from previous seminars, workshops, or lectures. You want to extract the knowledge from these videos to:
- Create course materials
- Generate lecture notes
- Build a knowledge base
- Repurpose content for different formats

### The Problem

**Manually transcribing video content is extremely time-consuming!**

Consider this: If you have 6 hours of video content, manual transcription could take:
- **4-5 person days** of work (for typing)
- Additional time for proofreading and formatting
- Risk of transcription errors and inconsistencies

This is simply not scalable or cost-effective.

### The Solution

We'll **automate this entire process** using OpenAI's powerful AI models:
1. **Whisper** - For accurate speech-to-text transcription
2. **GPT-5-nano** - For cleaning and formatting the transcripts

What would take days manually can be completed in minutes!

### About the Video

**In this notebook, we'll work with 1 example video to demonstrate the complete workflow.** This allows you to see the entire process from start to finish, understand each step, and then apply it to larger video collections.

---

Let's get started!

---

# Setup

## Install Dependencies

We'll need several Python packages:
- **openai** - To interact with OpenAI's Whisper and GPT models
- **moviepy** - To extract audio from video files
- **gdown** - To download files from Google Drive

In [ ]:
# Install required packages
!pip install -q openai==2.28.0 moviepy==1.0.3 gdown==5.2.1

# Suppress deprecation warnings for cleaner output
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)

print("✅ All dependencies installed!")

## API Key Configuration

You have two methods to provide your API key:

**Method 1 (Recommended)**: Use Colab Secrets
1. Click the 🔑 icon in the left sidebar
2. Click "Add new secret"
3. Name: `OPENAI_API_KEY`
4. Value: Your OpenAI API key
5. Enable notebook access

**Method 2 (Fallback)**: Manual input when prompted

Run the cell below to configure authentication:

In [ ]:
import os

# Configure OpenAI API key
# Method 1: Try to get API key from Colab secrets (recommended)
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    # Method 2: Manual input (fallback)
    from getpass import getpass
    print("💡 To use Colab secrets: Go to 🔑 (left sidebar) → Add new secret → Name: OPENAI_API_KEY")
    OPENAI_API_KEY = getpass("Enter your OpenAI API Key: ")

# Set the API key as an environment variable
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Validate that the API key is set
if not OPENAI_API_KEY or OPENAI_API_KEY.strip() == "":
    raise ValueError("❌ ERROR: No API key provided!")

print("✅ Authentication configured!")

# Configure which OpenAI model to use
# Options: "gpt-4o", "gpt-4o-mini", "gpt-4-turbo", "gpt-3.5-turbo", "gpt-5-nano", etc.
OPENAI_MODEL = "gpt-5-nano"  # Using gpt-5-nano for cost efficiency
print(f"Selected Model: {OPENAI_MODEL}")

## Import Required Libraries

In [ ]:
# Import necessary libraries
import os
from pathlib import Path
from openai import OpenAI
from moviepy.editor import VideoFileClip
import gdown

print("✅ All libraries imported successfully!")

## Initialize OpenAI Client

Now let's create a client instance to interact with the OpenAI API:

In [ ]:
# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ OpenAI client initialized successfully!")

---

## Planning Our Work

Before diving into the code, let's break down this task into **three clear steps**:

### Step 1: Convert Videos to Audio Format Only
Video files contain both visual and audio information, but we only need the audio for transcription.

### Step 2: Send Audio Files to Whisper Model to Obtain Transcriptions
Whisper is OpenAI's state-of-the-art speech-to-text model. It will convert our audio files into text transcripts automatically.

### Step 3: Use GPT to Clean These Transcriptions
Raw transcripts from speech-to-text models often contain:
- Filler words ("um", "uh", "you know")
- Run-on sentences without proper punctuation
- Repetitions and false starts
- Lack of proper formatting

GPT models excel at text refinement and can transform these raw transcripts into clean, professional lecture notes.

---

### Creating Working Directories

Due to the breakdown into these three steps, let's **create directories to store created audio files and intermediate transcriptions**. This keeps our workflow organized and makes it easy to review outputs at each stage.

## File Paths and Working Directories

Before diving into the code, let's set up our file paths and create working directories:
- **Input:** Videos in `/content/videos/`
- **Intermediate:** Audio files in `/content/audio_files/`
- **Intermediate:** Raw transcripts in `/content/transcripts/`
- **Output:** Cleaned transcripts in `/content/cleaned_transcripts/`

In [ ]:
# Define file paths
input_folder_path = '/content/videos'
audio_files_dir = '/content/audio_files'
transcripts_dir = '/content/transcripts'
cleaned_dir = '/content/cleaned_transcripts'

# Create necessary directories if they don't exist
os.makedirs(input_folder_path, exist_ok=True)
os.makedirs(audio_files_dir, exist_ok=True)
os.makedirs(transcripts_dir, exist_ok=True)
os.makedirs(cleaned_dir, exist_ok=True)

print("✅ File paths configured and directories created:")
print(f"  📥 Input videos: {input_folder_path}")
print(f"  🔊 Audio files: {audio_files_dir}")
print(f"  📝 Raw transcripts: {transcripts_dir}")
print(f"  ✨ Cleaned transcripts: {cleaned_dir}")

## Download Video from Google Drive

We'll download the example video directly from Google Drive using the gdown library. The video will be saved to `/content/videos/` automatically.

In [ ]:
# Google Drive file ID for the example video
file_id = "1WH_fRbD-qDWg8Hj7U3Ey0o5D7I4it_Ep"
filename = "example_video.mp4"
output_path = os.path.join(input_folder_path, filename)

print("Downloading video from Google Drive...\n")

# Check if video already exists
if os.path.exists(output_path):
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"✅ {filename} already exists ({file_size_mb:.2f} MB), skipping download")
else:
    url = f"https://drive.google.com/uc?id={file_id}"
    try:
        gdown.download(url, output_path, quiet=False)
        if os.path.exists(output_path):
            file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
            print(f"\n✅ Downloaded: {filename} ({file_size_mb:.2f} MB)")
        else:
            print(f"❌ Failed to download: {filename}")
    except Exception as e:
        print(f"❌ Error downloading {filename}: {str(e)}")

# Verify
video_files = [f for f in os.listdir(input_folder_path) if f.endswith('.mp4')]
print(f"\n✅ Found {len(video_files)} video file(s) ready for processing")

---

## Step 1: Video to Audio Conversion

### Why Audio-Only?

**Video files are large and contain visual information we don't need.** When transcribing speech, only the audio track matters. By extracting just the audio:
- Audio files are **significantly smaller** than videos (typically 10-20x smaller)
- Processing is **cheaper** (less data to transfer and store)
- Whisper **only analyzes audio** anyway, so we're not losing any information

For example, a 100MB video might contain only 5-10MB of audio data. Why send 90MB of unnecessary visual data to the API?

### 💡 Key Points:
- We use `moviepy` library to extract audio from video files
- Videos are in `.mp4` format
- Audio files are saved as `.mp3` (compressed, efficient format)
- Each video gets a corresponding audio file with the same base name

In [ ]:
print("🎵 Step 1: Converting videos to audio...\n")

# Loop through all files in the input folder
for filename in os.listdir(input_folder_path):
    # Check if the file is a video file (.mp4)
    if filename.endswith('.mp4'):
        video_path = os.path.join(input_folder_path, filename)

        # Create audio filename (replace .mp4 with .mp3)
        audio_filename = filename.replace('.mp4', '.mp3')
        audio_path = os.path.join(audio_files_dir, audio_filename)

        try:
            print(f"  🔄 Processing: {filename}")

            # Load the video file
            video_clip = VideoFileClip(video_path)

            # Extract audio from video
            audio_clip = video_clip.audio

            # Save audio as MP3 file
            audio_clip.write_audiofile(audio_path, verbose=False, logger=None)

            # Close clips to free up resources
            audio_clip.close()
            video_clip.close()

            print(f"  ✅ Audio extracted: {audio_filename}\n")

        except Exception as e:
            print(f"  ❌ Error processing {filename}: {str(e)}\n")
            continue

print("\n✅ Step 1 Complete: All videos converted to audio files!")
print(f"   Audio files saved to: {audio_files_dir}")

---

## Step 2: Speech-to-Text via Whisper

### What is Whisper?

**Whisper is OpenAI's state-of-the-art automatic speech recognition (ASR) system.** It was trained on 680,000 hours of multilingual and multitask supervised data collected from the web. This massive training dataset leads to:
- **High accuracy** across different accents and speaking styles
- **Multilingual support** for 98 languages
- **Robust performance** even with background noise or audio quality issues

### Whisper API Documentation

The Whisper API accepts audio files and returns transcriptions. Here are the key parameters:

- **`model`**: `"whisper-1"` (the current Whisper model version)
- **`file`**: Audio file object (supports mp3, mp4, wav, and more)
- **`response_format`**: `'text'` returns plain text, `'json'` returns detailed JSON with timestamps
- **File size limit**: 25 MB (for larger files, split them into chunks)
- **Supported formats**: mp3, mp4, mpeg, mpga, m4a, wav, webm

### 💡 Key Points:
- Whisper automatically detects the language (no need to specify)
- It adds punctuation and formatting automatically
- Processing time is typically 20-30% of the audio duration
- We save raw transcripts to review before cleaning

In [ ]:
print("🎤 Step 2: Transcribing audio files with Whisper...\n")

# Verify transcripts directory exists
if not os.path.exists(transcripts_dir):
    os.makedirs(transcripts_dir, exist_ok=True)
    print(f"📁 Created transcripts directory: {transcripts_dir}\n")

# Check if audio files exist
audio_files = [f for f in os.listdir(audio_files_dir) if f.endswith('.mp3')]
if not audio_files:
    print(f"⚠️ No audio files found in {audio_files_dir}")
    print("💡 Please run Step 1 first to convert videos to audio.")
else:
    print(f"Found {len(audio_files)} audio file(s) to transcribe.\n")

    # Loop through all audio files
    for filename in audio_files:
        audio_path = os.path.join(audio_files_dir, filename)

        # Create transcript filename (replace .mp3 with _transcript.txt)
        transcript_filename = filename.replace('.mp3', '_transcript.txt')
        transcript_path = os.path.join(transcripts_dir, transcript_filename)

        try:
            print(f"  🔄 Transcribing: {filename}")
            print(f"     Input: {audio_path}")
            print(f"     Output: {transcript_path}")

            # Open audio file and send to Whisper API
            with open(audio_path, 'rb') as audio_file:
                transcript_text = client.audio.transcriptions.create(
                    model="whisper-1",
                    file=audio_file,
                    response_format='text'
                )

            # Save the transcript to a text file
            with open(transcript_path, 'w', encoding='utf-8') as transcript_file:
                transcript_file.write(transcript_text)

            # Verify file was saved
            if os.path.exists(transcript_path):
                file_size = os.path.getsize(transcript_path)
                print(f"  ✅ Transcript saved: {transcript_filename} ({file_size:,} bytes)")
                print(f"     Preview: {transcript_text[:100]}...\n")
            else:
                print(f"  ❌ Failed to save transcript: {transcript_filename}\n")

        except Exception as e:
            print(f"  ❌ Error transcribing {filename}: {str(e)}\n")
            continue

    print("\n✅ Step 2 Complete: All audio files transcribed!")
    print(f"   Transcripts saved to: {transcripts_dir}")

    # Show all saved transcripts
    saved_transcripts = [f for f in os.listdir(transcripts_dir) if f.endswith('_transcript.txt')]
    print(f"   Total transcripts saved: {len(saved_transcripts)}")

---

## Step 3: Cleaning Transcriptions via GPT

### Why Clean Transcripts?

**Whisper captures speech accurately but includes imperfections that make transcripts hard to read and use.** Raw speech-to-text output often contains:

- **Filler words**: "um", "uh", "you know", "like", "so"
- **Run-on sentences**: Lack of proper sentence breaks and punctuation
- **Repetitions**: People often repeat words or rephrase thoughts mid-sentence
- **False starts**: Beginning a sentence one way, then starting over
- **Poor formatting**: Missing paragraph breaks, inconsistent capitalization

**GPT models excel at text refinement.** They can transform raw transcripts into clean, professional lecture notes that are:
- Easy to read and understand
- Properly formatted with clear structure
- Free of distracting filler words
- Suitable for use as course materials


### 💡 Key Points:
- We use a carefully crafted prompt to guide the cleaning process
- The model is instructed to stay faithful to original content (no hallucinations)
- Results are saved as separate "cleaned" files for easy comparison

In [ ]:
# Define the cleaning task description
task_description = (
    "You are a helpful assistant tasked with cleaning up lecture notes. "
    "Make the text coherent, correct any typos, and format it for a lecturer "
    "to use as speaking notes. Keep the text friendly, to the point, and ensure "
    "it does not deviate from the original content."
)

print("✨ Step 3: Cleaning transcripts with GPT-5-nano...\n")
print(f" Task: {task_description}\n")

# Verify cleaned directory exists
if not os.path.exists(cleaned_dir):
    os.makedirs(cleaned_dir, exist_ok=True)
    print(f" Created cleaned transcripts directory: {cleaned_dir}\n")

# Check if transcript files exist
transcript_files = [f for f in os.listdir(transcripts_dir) if f.endswith('_transcript.txt')]
if not transcript_files:
    print(f"⚠️ No transcript files found in {transcripts_dir}")
    print("💡 Please run Step 2 first to generate transcripts.")
else:
    print(f"Found {len(transcript_files)} transcript(s) to clean.\n")

    # Loop through all transcript files
    for filename in transcript_files:
        transcript_path = os.path.join(transcripts_dir, filename)

        # Create cleaned filename
        cleaned_filename = filename.replace('_transcript.txt', '_cleaned.txt')
        cleaned_path = os.path.join(cleaned_dir, cleaned_filename)

        try:
            print(f"  🔄 Cleaning: {filename}")
            print(f"     Input: {transcript_path}")
            print(f"     Output: {cleaned_path}")

            # Read the raw transcript
            with open(transcript_path, 'r', encoding='utf-8') as f:
                transcript_content = f.read()

            print(f"     Read {len(transcript_content):,} characters from raw transcript")

            # Create the input for GPT with task description and transcript
            input_text = f"{task_description}\n\nTranscript to clean:\n{transcript_content}"

            # Call GPT-5-nano API to clean the transcript
            response = client.responses.create(
                model=OPENAI_MODEL,
                input=input_text
            )

            # Extract the cleaned text from response
            cleaned_text = response.output_text

            # Save the cleaned transcript
            with open(cleaned_path, 'w', encoding='utf-8') as f:
                f.write(cleaned_text)

            # Verify file was saved
            if os.path.exists(cleaned_path):
                file_size = os.path.getsize(cleaned_path)
                print(f"  ✅ Cleaned transcript saved: {cleaned_filename} ({file_size:,} bytes)")
                print(f"     Preview: {cleaned_text[:100]}...\n")
            else:
                print(f"  ❌ Failed to save cleaned transcript: {cleaned_filename}\n")

        except Exception as e:
            print(f"  ❌ Error cleaning {filename}: {str(e)}\n")
            continue

    print("\n✅ Step 3 Complete: All transcripts cleaned!")
    print(f"   Cleaned transcripts saved to: {cleaned_dir}")

    # Show all saved cleaned transcripts
    saved_cleaned = [f for f in os.listdir(cleaned_dir) if f.endswith('_cleaned.txt')]
    print(f"   Total cleaned transcripts saved: {len(saved_cleaned)}")

---

## Results Comparison

Let's compare the raw transcript from Whisper with the cleaned version from GPT to see the improvement!


When comparing raw transcripts to cleaned versions, you'll typically see these improvements:

- **Removed filler words**: "um", "uh", "like", "you know", "stuff"
- **Better structure**: Clear paragraphs and formatting
- **Professional tone**: More suitable for lecture notes
- **Enhanced readability**: Easier to follow and understand
- **Same content**: No deviation from the original meaning

Let's view your actual results below:

### Before vs. After: Your Actual Transcripts

Here's a side-by-side comparison of your first video's transcript:

In [ ]:
# Get the first transcript file for comparison
transcript_files = [f for f in os.listdir(transcripts_dir) if f.endswith('_transcript.txt')]

if transcript_files:
    # Read the first raw transcript
    sample_transcript = transcript_files[0]
    transcript_path = os.path.join(transcripts_dir, sample_transcript)

    with open(transcript_path, 'r', encoding='utf-8') as f:
        raw_text = f.read()

    # Read the corresponding cleaned transcript
    cleaned_sample = sample_transcript.replace('_transcript.txt', '_cleaned.txt')
    cleaned_path = os.path.join(cleaned_dir, cleaned_sample)

    with open(cleaned_path, 'r', encoding='utf-8') as f:
        cleaned_text = f.read()

    # Display comparison
    print("=" * 80)
    print(f" File: {sample_transcript}")
    print("=" * 80)

    print("\n" + "🔍 RAW TRANSCRIPT (from Whisper)".center(80))
    print("-" * 80)
    print(raw_text[:800])  # Show first 800 characters
    if len(raw_text) > 800:
        print("\n... (showing first 800 characters)")

    print("\n" + "=" * 80)

    print("\n" + "✨ CLEANED TRANSCRIPT (from GPT-5-nano)".center(80))
    print("-" * 80)
    print(cleaned_text[:800])  # Show first 800 characters
    if len(cleaned_text) > 800:
        print("\n... (showing first 800 characters)")

    print("\n" + "=" * 80)

    # Show statistics
    print("\n Statistics:")
    print(f"  • Raw transcript length: {len(raw_text):,} characters")
    print(f"  • Cleaned transcript length: {len(cleaned_text):,} characters")
    print(f"  • Character reduction: {len(raw_text) - len(cleaned_text):,} characters ({((len(raw_text) - len(cleaned_text)) / len(raw_text) * 100):.1f}%)")

else:
    print("⚠️ No transcripts found to display.")
    print("💡 Make sure you've run Steps 2 and 3 first to generate transcripts.")

### Download Your Cleaned Transcripts

You can download all the cleaned transcripts to your local machine:

In [ ]:
from google.colab import files
import zipfile

# Create a zip file with all cleaned transcripts
zip_filename = '/content/cleaned_transcripts.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for filename in os.listdir(cleaned_dir):
        if filename.endswith('.txt'):
            file_path = os.path.join(cleaned_dir, filename)
            zipf.write(file_path, filename)

print("Created zip file with all cleaned transcripts")
print("⬇️ Downloading...")
files.download(zip_filename)
print("✅ Download complete!")

---

## Limitations and Considerations

While this workflow is powerful and automated, it's important to understand its limitations:

### Whisper Limitations:

1. **Audio Quality Matters**
   - Best results with clear speech and minimal background noise
   - Struggles with heavy accents, mumblings, or poor recording quality
   - Multiple overlapping speakers can cause confusion

2. **File Size Constraint**
   - **25MB file size limit** per API request
   - Longer audio files (typically >1 hour of good quality audio) need to be split into chunks
   - For production use, implement audio chunking logic

3. **Language Detection**
   - While Whisper supports 98 languages, accuracy varies by language
   - English has the highest accuracy due to more training data
   - Code-switching (mixing languages) can be challenging

### GPT Cleaning Limitations:

1. **Context Length**
   - Very long transcripts may exceed model context limits
   - May need to process in sections for multi-hour videos

2. **Potential Over-Editing**
   - Model might occasionally rephrase content too much
   - Important to review critical transcripts manually

3. **Domain-Specific Terms**
   - May occasionally misinterpret technical jargon or specialized terminology
   - Consider adding domain-specific instructions to the prompt

### ✅ Best Practices:

- **Always review critical content** - Don't rely solely on automated processing for important materials
- **Test with sample videos** first to ensure quality meets your needs
- **Use good source material** - Better input quality = better output
- **Keep original files** - Don't delete raw transcripts; they're useful for comparison
- **Iterate on prompts** - Customize the cleaning prompt for your specific use case



❌ This workflow is not ideal for: Multi-speaker debates, heavily accented speech, very low-quality audio, or content with critical legal/medical importance requiring 100% accuracy.

---



### Additional Resources:

- [OpenAI Whisper Documentation](https://platform.openai.com/docs/guides/speech-to-text)
- [OpenAI GPT Documentation](https://platform.openai.com/docs/guides/text-generation)
- [MoviePy Documentation](https://zulko.github.io/moviepy/)

